# ISFEST 2026 DATA COMPETITION (UNIVERSITAS MULTIMEDIA NUSANTARA)
## Studi Kasus: Mapping the Demand for Electric Vehicle (EV) Charging Infrastructure
### Kerangka Kerja: Cross-Industry Standard Process for Data Mining (CRISP-DM)
### Pendekatan Pemodelan: Tri-Model Ensemble (LightGBM + CatBoost + XGBoost)

---
**Identitas Tim:**
* **Nama Tim:** `[NamaTim]`
* **Anggota Tim:**
  1. [Nama Anggota 1] (Ketua)
  2. [Nama Anggota 2]
  3. [Nama Anggota 3]
* **Institusi:** Universitas Negeri Surabaya (UNESA)
* **Deskripsi Singkat:** Notebook ini memuat alur analitis dan pemodelan prediktif terstruktur berbasis CRISP-DM untuk memprediksi tingkat okupansi fasilitas pengisian daya kendaraan listrik (`utilization_rate`), menganalisis faktor penentu permintaan, serta merumuskan rekomendasi bisnis yang selaras dengan target **SDG 7: Affordable and Clean Energy**.
---

# 1. Business Understanding (CRISP-DM Fase 1)

### 1.1 Latar Belakang Domain
Perkembangan kendaraan bermotor listrik (*Electric Vehicle* / EV) merupakan langkah strategis dalam dekarbonisasi sektor transportasi global. Namun, keberhasilan adopsi EV sangat bergantung pada keandalan dan ketersediaan infrastruktur pengisian daya (*charging stations*). Di lapangan, terdapat beberapa permasalahan operasional utama:
1. **Fenomena Range Anxiety:** Pengemudi EV kerap menghadapi kekhawatiran kehabisan daya baterai sebelum menjangkau stasiun pengisian daya yang tersedia dan beroperasi normal.
2. **Ketimpangan Pemanfaatan (Imbalance Demand):** Sebagian stasiun di simpul transportasi utama mengalami antrean panjang, sedangkan stasiun di lokasi lain mengalami tingkat keterpakaian rendah (*underutilized*), yang berujung pada inefisiensi investasi modal bagi penyedia layanan (*Charge Point Operator* / CPO).
3. **Faktor Eksternal Dinamis:** Tingkat pemanfaatan stasiun dipengaruhi secara simultan oleh dimensi temporal (jam sibuk vs jam santai), karakteristik infrastruktur (tipe konektor dan kapasitas daya kW), kondisi cuaca lokal, serta fluktuasi harga bahan bakar konvensional sebagai pembanding.

### 1.2 Tujuan Analisis dan Metrik Keberhasilan
* **Tujuan Prediksi:** Mengembangkan model *machine learning regression* ensemble untuk memprediksi tingkat pemanfaatan stasiun pengisian daya (`utilization_rate`, skala 0.0 hingga 1.0) dengan interval pencatatan 30 menit.
* **Metrik Evaluasi Utama:** Sesuai ketentuan panitia ISFEST 2026, akurasi dievaluasi menggunakan *Root Mean Squared Error* (RMSE), didukung oleh metrik *Mean Absolute Error* (MAE) dan koefisien determinasi ($R^2$).
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
* **Tujuan Strategis:** Mengidentifikasi variabel kunci penentu lonjakan permintaan dan menghasilkan usulan kebijakan operasional serta strategi penetapan tarif dinamis (*dynamic pricing*) yang aplikatif.

### 1.3 Penyelarasan dengan Sustainable Development Goals (SDG) 7
Solusi yang dikembangkan secara langsung mendukung pencapaian **SDG 7 (Affordable and Clean Energy)**:
* **Target 7.1:** Memastikan akses energi yang andal dan terjangkau melalui optimalisasi ketersediaan titik pengisian daya publik.
* **Target 7.2:** Mendukung integrasi energi terbarukan melalui manajemen beban puncak (*peak load balancing*) pengisian daya kendaraan listrik.
* **Target 7.3:** Menggandakan tingkat efisiensi energi global dengan meminimalkan waktu tunggu dan waktu menganggur (*idle capacity*) pada stasiun pengisian daya.

# 2. Data Understanding (CRISP-DM Fase 2)

Pada fase ini, dilakukan inisialisasi pustaka, deteksi akselerator GPU P100, pemuatan dataset pelatihan (`train.csv`) dan dataset pengujian (`test.csv`), reduksi penggunaan memori, serta pemeriksaan anomali data sesuai petunjuk teknis dewan juri.

In [ ]:
# Inisialisasi Pustaka dan Konfigurasi Lingkungan
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deteksi Hardware & Akselerator GPU P100
import torch

# Machine Learning, Boosting & Optimasi
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Standar visualisasi profesional
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#bbbbbb'
plt.rcParams['axes.linewidth'] = 0.8

FIGURES_DIR = './figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

TEAM_NAME = "NamaTim"
SEED = 42

# Konfigurasi Akselerator GPU
USE_GPU = torch.cuda.is_available()
if USE_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"[INFO] Akselerator GPU Terdeteksi: {gpu_name} (Siap digunakan untuk CatBoost & XGBoost)")
else:
    print("[INFO] Akselerator GPU tidak aktif. Berjalan pada Multi-Core CPU Threads.")

print(f"[INFO] Inisialisasi lingkungan selesai. Folder visualisasi: '{FIGURES_DIR}'")

### 2.1 Pemuatan Data Fleksibel (Kompatibel Lokal dan Kaggle)
Fungsi `find_file` secara otomatis memprioritaskan path dataset Kaggle Anda `/kaggle/input/datasets/rabbaniyuki/isfest-dataset/`.

In [ ]:
# Konfigurasi Lokasi Dataset (Prioritas Kaggle Path & Lokal)
KAGGLE_DATA_DIR = '/kaggle/input/datasets/rabbaniyuki/isfest-dataset'

def find_file(filename):
    kaggle_path = os.path.join(KAGGLE_DATA_DIR, filename)
    if os.path.exists(kaggle_path):
        return kaggle_path
    if os.path.exists(filename):
        return filename
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    return kaggle_path

# Fungsi Optimasi Tipe Data (Penghematan RAM)
def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not str(col_type).startswith('datetime'):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"[INFO] Pengurangan memori: {start_mem:.1f} MB -> {end_mem:.1f} MB (Hemat: {100*(start_mem-end_mem)/start_mem:.1f}%)")
    return df

train_path = find_file('train.csv')
test_path = find_file('test.csv')

print(f"Membaca data pelatihan dari: {train_path}")
train_raw = pd.read_csv(train_path)
print(f"Membaca data pengujian dari: {test_path}")
test_raw = pd.read_csv(test_path)

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)

print(f"[INFO] Data Pelatihan: {train_raw.shape[0]:,} baris, {train_raw.shape[1]} kolom.")
print(f"[INFO] Data Pengujian : {test_raw.shape[0]:,} baris, {test_raw.shape[1]} kolom.")

### 2.2 Audit Anomali Data Sesuai Panduan Soal
Berdasarkan petunjuk resmi dewan juri pada lembar soal kompetisi, dilakukan pemeriksaan atas tiga petunjuk khusus:
1. **Pembedaan Stasiun dengan Nama Sama:** Memastikan bahwa entitas pemodelan menggunakan `station_id` (kunci unik), bukan `station_name`.
2. **Missing Values Fitur Cuaca:** Mengidentifikasi sebaran nilai kosong pada variabel suhu udara dan presipitasi.
3. **Celah Waktu (Data Gap) 31 Desember 2025:** Menemukan batas pencatatan waktu pada data uji guna mencegah galat agregasi temporal.

In [ ]:
# 1. Audit Kesamaan Nama Stasiun dengan ID Berbeda
station_mapping = train_raw.groupby('station_name')['station_id'].nunique()
duplicate_names = station_mapping[station_mapping > 1]

print("=== [AUDIT 1] Stasiun Bernama Sama dengan ID Berbeda ===")
for st_name, id_count in duplicate_names.items():
    st_ids = train_raw[train_raw['station_name'] == st_name]['station_id'].unique().tolist()
    print(f"Nama Stasiun: '{st_name}' mencakup {id_count} ID berbeda: {st_ids}")
print("Hasil: station_id ditetapkan sebagai entitas unik primer pemodelan.\n")

# 2. Audit Nilai Kosong (Missing Values)
print("=== [AUDIT 2] Distribusi Nilai Kosong ===")
null_train = train_raw.isnull().sum()[train_raw.isnull().sum() > 0]
null_test = test_raw.isnull().sum()[test_raw.isnull().sum() > 0]
null_df = pd.DataFrame({'Train Nulls': null_train, 'Test Nulls': null_test})
print(null_df)
print("Hasil: Nilai kosong hanya terdapat pada 'temperature_f' dan 'precipitation_mm'.\n")

# 3. Audit Celah Waktu Tanggal 31 Desember 2025 pada Data Uji
test_raw['dt_audit'] = pd.to_datetime(test_raw['timestamp'], format='mixed')
dec31_data = test_raw[test_raw['dt_audit'].dt.date == pd.to_datetime('2025-12-31').date()]
print("=== [AUDIT 3] Kontinuitas Data Uji pada 31 Desember 2025 ===")
print(f"Jumlah baris 31 Desember 2025: {len(dec31_data)} baris.")
print(f"Jam yang tercatat: {dec31_data['dt_audit'].dt.hour.unique().tolist()} (Hanya jam 00:00).")
print("Hasil: Terkonfirmasi adanya celah waktu pada 31 Desember yang perlu diantisipasi.")
test_raw.drop(columns=['dt_audit'], inplace=True)

### 2.3 Exploratory Data Analysis (EDA) dan Analisis Karakteristik
Eksplorasi dilakukan untuk memahami sebaran target variabel, pola fluktuasi diurnal 24 jam, pengaruh jenis charger, serta dampak kondisi cuaca dan harga bensin lokal.

In [ ]:
# Ekstraksi Komponen Waktu untuk Eksplorasi Visual
train_raw['datetime'] = pd.to_datetime(train_raw['timestamp'], format='mixed')
test_raw['datetime'] = pd.to_datetime(test_raw['timestamp'], format='mixed')

train_raw['hour'] = train_raw['datetime'].dt.hour
train_raw['dayofweek'] = train_raw['datetime'].dt.dayofweek
train_raw['is_weekend'] = train_raw['dayofweek'].isin([5, 6]).astype(int)

# Visualisasi 1: Distribusi Target utilization_rate
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(train_raw['utilization_rate'], bins=40, kde=True, color='#1d4ed8', ax=ax[0])
ax[0].set_title('Distribusi Probabilitas Target (utilization_rate)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Utilization Rate (0.0 - 1.0)')
ax[0].set_ylabel('Frekuensi')

sns.boxplot(x=train_raw['utilization_rate'], color='#93c5fd', ax=ax[1])
ax[1].set_title('Sebaran Boxplot Target (utilization_rate)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Utilization Rate')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_target_distribution.png'), dpi=300)
plt.show()

# Visualisasi 2: Kurva Utilisasi 24 Jam (Hari Kerja vs Akhir Pekan)
hourly_pattern = train_raw.groupby(['hour', 'is_weekend'])['utilization_rate'].mean().reset_index()
hourly_pattern['Kategori Hari'] = hourly_pattern['is_weekend'].map({0: 'Hari Kerja (Senin-Jumat)', 1: 'Akhir Pekan (Sabtu-Minggu)'})

plt.figure(figsize=(12, 4.8))
sns.lineplot(data=hourly_pattern, x='hour', y='utilization_rate', hue='Kategori Hari',
             marker='o', palette=['#1e40af', '#d97706'], linewidth=2.2)
plt.title('Kurva Fluktuasi Rata-rata Utilisasi 24 Jam (Weekday vs Weekend)', fontsize=13, fontweight='bold')
plt.xlabel('Jam Pengisian Daya (0 - 23)', fontsize=10)
plt.ylabel('Rata-rata Utilization Rate', fontsize=10)
plt.xticks(range(0, 24))
plt.axvspan(7, 9, color='#fee2e2', alpha=0.45, label='Jam Sibuk Pagi (07.00 - 09.00)')
plt.axvspan(16, 19, color='#fef3c7', alpha=0.45, label='Jam Sibuk Sore (16.00 - 19.00)')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_diurnal_hourly_pattern.png'), dpi=300)
plt.show()

In [ ]:
# Visualisasi 3: Pengaruh Tipe Charger dan Lingkungan Lokasi
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

charger_order = train_raw.groupby('charger_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='charger_type', y='utilization_rate', order=charger_order,
            palette='Blues_r', errorbar=None, ax=ax[0])
ax[0].set_title('Rata-rata Utilisasi Berdasarkan Tipe Charger', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Tipe Charger')
ax[0].set_ylabel('Rata-rata Utilization Rate')
ax[0].tick_params(axis='x', rotation=15)

loc_order = train_raw.groupby('location_type')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='location_type', y='utilization_rate', order=loc_order,
            palette='Purples_r', errorbar=None, ax=ax[1])
ax[1].set_title('Rata-rata Utilisasi Berdasarkan Lokasi Stasiun', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Tipe Lokasi')
ax[1].set_ylabel('Rata-rata Utilization Rate')
ax[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_infrastructure_analysis.png'), dpi=300)
plt.show()

# Visualisasi 4: Pengaruh Cuaca dan Rentang Harga Bensin
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

weather_order = train_raw.groupby('weather_condition')['utilization_rate'].mean().sort_values(ascending=False).index
sns.barplot(data=train_raw, x='weather_condition', y='utilization_rate', order=weather_order,
            palette='Spectral', errorbar=None, ax=ax[0])
ax[0].set_title('Utilisasi Berdasarkan Kondisi Cuaca', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Kondisi Cuaca')
ax[0].set_ylabel('Rata-rata Utilization Rate')
ax[0].tick_params(axis='x', rotation=20)

train_raw['gas_price_bin'] = pd.qcut(train_raw['gas_price_per_gallon'], q=5)
sns.barplot(data=train_raw, x='gas_price_bin', y='utilization_rate', palette='Greens', errorbar=None, ax=ax[1])
ax[1].set_title('Pengaruh Rentang Harga Bensin Lokal ($/Galon)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Kuintil Harga Bensin ($)')
ax[1].set_ylabel('Rata-rata Utilization Rate')
ax[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_weather_gas_impact.png'), dpi=300)
plt.show()

train_raw.drop(columns=['gas_price_bin'], inplace=True)

# 3. Data Preparation (CRISP-DM Fase 3)

Pada tahap persiapan data, langkah-langkah rekayasa fitur dilakukan secara komprehensif:
1. **Imputasi Temporal Missing Values:** Menerapkan forward-fill dan backward-fill kronologis per `station_id` untuk fitur suhu udara (`temperature_f`) dan presipitasi (`precipitation_mm`) sesuai arahan dewan juri. Sisa nilai kosong diisi menggunakan median kota.
2. **Fitur Temporal dan Siklis:** Mengekstraksi komponen jam, menit, hari, bulan, status akhir pekan, jam sibuk pagi (07.00 - 09.00), jam sibuk sore (16.00 - 19.00), serta transformasi trigonometri sinus dan kosinus.
3. **Multi-Hot Encoding Fasilitas Sekitar (Amenities):** Mengonversi atribut teks `amenities_nearby` menjadi fitur biner individual (WiFi, Restroom, Dining, Mall, Hotel, Park, Grocery, Convenience).
4. **Interaksi Hardware dan Kapasitas:** Menghitung daya per port (`power_per_port`), total kapasitas energi stasiun (`total_capacity_kw`), status pengisian daya ultra-cepat (`is_ultra_fast`), serta tarif gratis (`is_free_pricing`).
5. **Target Encoding Historis per Stasiun:** Memetakan rata-rata utilisasi historis stasiun secara aman dari data pelatihan untuk mencegah *data leakage*.

In [ ]:
# Penggabungan untuk Ekstraksi Fitur Terpadu
df_all = pd.concat([train_raw.drop(columns=['utilization_rate']), test_raw], axis=0, ignore_index=True)
df_all = df_all.sort_values(by=['station_id', 'datetime']).reset_index(drop=True)

# 1. Imputasi Temporal Nilai Kosong per Stasiun
print("[INFO] Melakukan imputasi temporal terarah...")
df_all['temperature_f'] = df_all.groupby('station_id')['temperature_f'].ffill().bfill()
df_all['precipitation_mm'] = df_all.groupby('station_id')['precipitation_mm'].ffill().bfill()

df_all['temperature_f'] = df_all.groupby('city')['temperature_f'].transform(lambda x: x.fillna(x.median()))
df_all['precipitation_mm'] = df_all.groupby('city')['precipitation_mm'].transform(lambda x: x.fillna(0.0))

# 2. Rekayasa Fitur Temporal
df_all['hour'] = df_all['datetime'].dt.hour
df_all['minute'] = df_all['datetime'].dt.minute
df_all['time_slot'] = df_all['hour'] + df_all['minute'] / 60.0
df_all['dayofweek'] = df_all['datetime'].dt.dayofweek
df_all['day'] = df_all['datetime'].dt.day
df_all['month'] = df_all['datetime'].dt.month
df_all['is_weekend'] = df_all['dayofweek'].isin([5, 6]).astype(int)

# Penanda Jam Sibuk (Peak Hours)
df_all['is_morning_peak'] = df_all['hour'].isin([7, 8, 9]).astype(int)
df_all['is_evening_peak'] = df_all['hour'].isin([16, 17, 18, 19]).astype(int)
df_all['is_peak_hour'] = (df_all['is_morning_peak'] | df_all['is_evening_peak']).astype(int)

# Transformasi Trigonometri Siklis
df_all['sin_hour'] = np.sin(2 * np.pi * df_all['time_slot'] / 24.0).astype(np.float32)
df_all['cos_hour'] = np.cos(2 * np.pi * df_all['time_slot'] / 24.0).astype(np.float32)
df_all['sin_dow'] = np.sin(2 * np.pi * df_all['dayofweek'] / 7.0).astype(np.float32)
df_all['cos_dow'] = np.cos(2 * np.pi * df_all['dayofweek'] / 7.0).astype(np.float32)

# 3. Interaksi Fitur Hardware & Infrastruktur
df_all['power_per_port'] = (df_all['power_output_kw'] / df_all['ports_total'].replace(0, 1)).astype(np.float32)
df_all['total_capacity_kw'] = (df_all['power_output_kw'] * df_all['ports_total']).astype(np.float32)
df_all['is_ultra_fast'] = (df_all['power_output_kw'] >= 150.0).astype(int)
df_all['is_free_pricing'] = (df_all['pricing_type'].str.lower() == 'free').astype(int)

# 4. Multi-Hot Encoding Fasilitas (Amenities)
amenities_list = ['WiFi', 'Restroom', 'Restaurant', 'Shopping Mall', 'Hotel', 'Park', 'Grocery Store', 'Convenience Store']
for amen in amenities_list:
    col_name = 'has_' + amen.lower().replace(' ', '_')
    df_all[col_name] = df_all['amenities_nearby'].fillna('').str.contains(amen, case=False).astype(int)
df_all['total_amenities'] = df_all[[c for c in df_all.columns if c.startswith('has_')]].sum(axis=1)

# Pemisahan Kembali Train dan Test dengan Menjaga Urutan Asli
train_features = df_all[df_all['id'].str.startswith('TRN_')].copy().reset_index(drop=True)
test_features = df_all[df_all['id'].str.startswith('TST_')].copy().reset_index(drop=True)

# Urutkan test_features persis sesuai indeks asli test_raw
test_features = test_raw[['id']].merge(test_features, on='id', how='left')

# Gabungkan target utilization_rate pada data pelatihan
train_features = train_features.merge(train_raw[['id', 'utilization_rate']], on='id', how='left')

print(f"[INFO] Rekayasa fitur selesai. Jumlah fitur: {train_features.shape[1]}")

### 3.1 Agregasi Historis per Stasiun (Tanpa Kebocoran Informasi)
Menghitung rata-rata dan deviasi standar tingkat pemanfaatan historis per stasiun serta kombinasi stasiun dan jam pengisian daya.

In [ ]:
# Agregasi Historis per Stasiun dari Data Pelatihan
station_stats = train_features.groupby('station_id')['utilization_rate'].agg(
    station_util_mean='mean',
    station_util_std='std',
    station_util_median='median'
).reset_index()

station_hourly = train_features.groupby(['station_id', 'hour'])['utilization_rate'].agg(
    station_hourly_mean='mean'
).reset_index()

train_features = train_features.merge(station_stats, on='station_id', how='left')
train_features = train_features.merge(station_hourly, on=['station_id', 'hour'], how='left')

test_features = test_features.merge(station_stats, on='station_id', how='left')
test_features = test_features.merge(station_hourly, on=['station_id', 'hour'], how='left')

global_mean = train_features['utilization_rate'].mean()
test_features['station_util_mean'] = test_features['station_util_mean'].fillna(global_mean)
test_features['station_util_std'] = test_features['station_util_std'].fillna(train_features['station_util_std'].mean())
test_features['station_util_median'] = test_features['station_util_median'].fillna(train_features['station_util_median'].median())
test_features['station_hourly_mean'] = test_features['station_hourly_mean'].fillna(test_features['station_util_mean'])

print("[INFO] Agregasi historis berhasil dipetakan ke train dan test set.")

# 4. Modeling (CRISP-DM Fase 4)

### 4.1 Skema Validasi Kronologis (Out-of-Time Validation)
Untuk mengevaluasi model secara realistis, pembagian data validasi dilakukan secara kronologis:
* **Pelatihan (Train OOT):** 1 Juli 2025 sampai 10 November 2025 (~880.000 data).
* **Validasi (Val OOT):** 11 November 2025 sampai 24 November 2025 (~168.000 data).

Skema ini menjamin bahwa model diuji pada data waktu yang belum pernah dilihat sebelumnya, mencerminkan kondisi inferensi pada data uji.

In [ ]:
# Definisi Fitur dan Kategori
cat_cols = ['station_id', 'network', 'city', 'state', 'location_type', 'charger_type', 'pricing_type', 'weather_condition', 'local_event']

for c in cat_cols:
    train_features[c] = train_features[c].astype('category')
    test_features[c] = test_features[c].astype('category')

feature_cols = [
    'power_output_kw', 'ports_total', 'power_per_port', 'total_capacity_kw', 'is_ultra_fast', 'is_free_pricing',
    'temperature_f', 'precipitation_mm', 'gas_price_per_gallon',
    'hour', 'dayofweek', 'month', 'is_weekend', 'is_peak_hour', 'is_morning_peak', 'is_evening_peak',
    'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow',
    'has_wifi', 'has_restroom', 'has_restaurant', 'has_shopping_mall', 'has_hotel', 'has_park', 'has_grocery_store', 'has_convenience_store', 'total_amenities',
    'station_util_mean', 'station_util_std', 'station_util_median', 'station_hourly_mean'
] + cat_cols

val_cutoff = pd.to_datetime('2025-11-10 00:00:00')

train_mask = train_features['datetime'] < val_cutoff
val_mask = train_features['datetime'] >= val_cutoff

X_tr, y_tr = train_features.loc[train_mask, feature_cols], train_features.loc[train_mask, 'utilization_rate']
X_va, y_va = train_features.loc[val_mask, feature_cols], train_features.loc[val_mask, 'utilization_rate']

print(f"[INFO] Baris Pelatihan (Train OOT): {len(X_tr):,}")
print(f"[INFO] Baris Validasi  (Val OOT)  : {len(X_va):,}")

### 4.2 Pemodelan Acuan (Baseline Ridge Regression)
Model regresi linier dibangun sebagai baseline pembanding performa awal sesuai rubrik kompetisi.

In [ ]:
# Baseline Linear Model (Ridge Regression)
num_cols = [c for c in feature_cols if c not in cat_cols]

ridge = Ridge(alpha=1.0)
ridge.fit(X_tr[num_cols].fillna(0), y_tr)
baseline_preds = np.clip(ridge.predict(X_va[num_cols].fillna(0)), 0.0, 1.0)

baseline_rmse = np.sqrt(mean_squared_error(y_va, baseline_preds))
baseline_mae = mean_absolute_error(y_va, baseline_preds)
baseline_r2 = r2_score(y_va, baseline_preds)

print("=== HASIL MODEL BASELINE (Ridge Regression) ===")
print(f"RMSE Baseline : {baseline_rmse:.4f}")
print(f"MAE Baseline  : {baseline_mae:.4f}")
print(f"R2 Baseline   : {baseline_r2:.4f}")

### 4.3 Model 1: LightGBM Regressor dengan Optimasi Optuna
Pencarian hyperparameter optimal untuk algoritma **LightGBM Regressor** dilakukan secara terarah menggunakan Optuna sebanyak 15 iterasi (*trials*) dengan pemangkas cerdas (*MedianPruner*).

In [ ]:
# Optimasi Optuna untuk LightGBM
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'n_estimators': 300,
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 80),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    
    sample_size = min(250000, len(X_tr))
    sample_idx = np.random.choice(len(X_tr), size=sample_size, replace=False)
    X_sample, y_sample = X_tr.iloc[sample_idx], y_tr.iloc[sample_idx]
    
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_sample, y_sample,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
    )
    
    val_preds = model.predict(X_va)
    return np.sqrt(mean_squared_error(y_va, val_preds))

def log_trial(study, trial):
    print(f"[Optuna] Trial {trial.number+1:02d}/15 | Val RMSE: {trial.value:.4f} | Terbaik: {study.best_value:.4f}")

print("Memulai Optimasi Hyperparameter Optuna untuk LightGBM (15 Trials)...")
study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=15, callbacks=[log_trial])

print(f"\n[Optuna Selesai] Nilai RMSE Terbaik: {study.best_value:.4f}")
print("Parameter Terbaik:", study.best_params)

# Pelatihan Model 1: LightGBM dengan Parameter Terbaik
best_lgb_params = study.best_params.copy()
best_lgb_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 600,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
})

model_lgb = lgb.LGBMRegressor(**best_lgb_params)
t_start = time.time()
model_lgb.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)
print(f"[INFO] Pelatihan LightGBM selesai dalam: {time.time() - t_start:.2f} detik")

lgb_val_preds = np.clip(model_lgb.predict(X_va), 0.0, 1.0)
lgb_rmse = np.sqrt(mean_squared_error(y_va, lgb_val_preds))
lgb_mae = mean_absolute_error(y_va, lgb_val_preds)
lgb_r2 = r2_score(y_va, lgb_val_preds)

print(f"=== HASIL MODEL 1 (LightGBM Tuned) ===")
print(f"RMSE : {lgb_rmse:.4f} | MAE : {lgb_mae:.4f} | R2 : {lgb_r2:.4f}")

### 4.4 Model 2: CatBoost Regressor (Akselerasi GPU P100 Native)
CatBoost memiliki keunggulan inheren dalam penanganan variabel kategorikal (*ordered target statistics*) dan memanfaatkan akselerator GPU P100 secara *native*.

In [ ]:
# Pelatihan Model 2: CatBoost Regressor
cb_task_type = 'GPU' if USE_GPU else 'CPU'
print(f"[INFO] Memulai pelatihan CatBoost Regressor (Task Type: {cb_task_type})...")

cb_params = {
    'iterations': 500,
    'learning_rate': 0.09,
    'depth': 7,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': SEED,
    'task_type': cb_task_type,
    'verbose': 100
}

model_cb = cb.CatBoostRegressor(**cb_params)
t_start = time.time()
model_cb.fit(
    X_tr, y_tr,
    cat_features=cat_cols,
    eval_set=(X_va, y_va),
    early_stopping_rounds=30,
    verbose=False
)
print(f"[INFO] Pelatihan CatBoost selesai dalam: {time.time() - t_start:.2f} detik")

cb_val_preds = np.clip(model_cb.predict(X_va), 0.0, 1.0)
cb_rmse = np.sqrt(mean_squared_error(y_va, cb_val_preds))
cb_mae = mean_absolute_error(y_va, cb_val_preds)
cb_r2 = r2_score(y_va, cb_val_preds)

print(f"=== HASIL MODEL 2 (CatBoost Regressor) ===")
print(f"RMSE : {cb_rmse:.4f} | MAE : {cb_mae:.4f} | R2 : {cb_r2:.4f}")

### 4.5 Model 3: XGBoost Regressor (Akselerasi GPU P100 Native)
XGBoost menggunakan metode partisi histogram (*hist*) dan akselerasi CUDA pada GPU P100 untuk menghasilkan pohon keputusan yang presisi dan cepat.

In [ ]:
# Pelatihan Model 3: XGBoost Regressor
xgb_device = 'cuda' if USE_GPU else 'cpu'
print(f"[INFO] Memulai pelatihan XGBoost Regressor (Device: {xgb_device})...")

xgb_params = {
    'n_estimators': 500,
    'learning_rate': 0.08,
    'max_depth': 8,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'tree_method': 'hist',
    'device': xgb_device,
    'enable_categorical': True,
    'random_state': SEED,
    'n_jobs': -1
}

model_xgb = xgb.XGBRegressor(**xgb_params)
t_start = time.time()
model_xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    verbose=False
)
print(f"[INFO] Pelatihan XGBoost selesai dalam: {time.time() - t_start:.2f} detik")

xgb_val_preds = np.clip(model_xgb.predict(X_va), 0.0, 1.0)
xgb_rmse = np.sqrt(mean_squared_error(y_va, xgb_val_preds))
xgb_mae = mean_absolute_error(y_va, xgb_val_preds)
xgb_r2 = r2_score(y_va, xgb_val_preds)

print(f"=== HASIL MODEL 3 (XGBoost Regressor) ===")
print(f"RMSE : {xgb_rmse:.4f} | MAE : {xgb_mae:.4f} | R2 : {xgb_r2:.4f}")

### 4.6 Tri-Model Weighted Ensemble Blending
Menggabungkan ketiga model boosting unggulan (LightGBM, CatBoost, dan XGBoost) menggunakan pembobotan optimal (*weighted averaging*). Penggabungan model dengan arsitektur berbeda terbukti mampu meminimalkan variansi galat dan menghasilkan prediksi yang lebih stabil.

In [ ]:
# Penggabungan Prediksi Ensemble
w_lgb = 0.40
w_cb = 0.35
w_xgb = 0.25

ensemble_val_preds = np.clip(
    w_lgb * lgb_val_preds + w_cb * cb_val_preds + w_xgb * xgb_val_preds,
    0.0, 1.0
)

ensemble_rmse = np.sqrt(mean_squared_error(y_va, ensemble_val_preds))
ensemble_mae = mean_absolute_error(y_va, ensemble_val_preds)
ensemble_r2 = r2_score(y_va, ensemble_val_preds)

print("=== HASIL TRI-MODEL ENSEMBLE BLENDING ===")
print(f"Bobot Model  : LightGBM ({w_lgb*100:.0f}%) + CatBoost ({w_cb*100:.0f}%) + XGBoost ({w_xgb*100:.0f}%)")
print(f"RMSE Ensemble : {ensemble_rmse:.4f}")
print(f"MAE Ensemble  : {ensemble_mae:.4f}")
print(f"R2 Ensemble   : {ensemble_r2:.4f}")

# 5. Evaluation (CRISP-DM Fase 5)

Evaluasi komparatif komprehensif dilakukan untuk membandingkan performa seluruh model, menganalisis residual error, serta memeriksa kontribusi variabel terpenting.

In [ ]:
# 1. Tabel Komparasi Seluruh Model
eval_df = pd.DataFrame({
    'Model': [
        'Baseline (Ridge Regression)',
        'Model 1: LightGBM (Tuned)',
        'Model 2: CatBoost (GPU)',
        'Model 3: XGBoost (GPU)',
        'Champion: Tri-Model Ensemble'
    ],
    'RMSE': [baseline_rmse, lgb_rmse, cb_rmse, xgb_rmse, ensemble_rmse],
    'MAE': [baseline_mae, lgb_mae, cb_mae, xgb_mae, ensemble_mae],
    'R-Squared (R2)': [baseline_r2, lgb_r2, cb_r2, xgb_r2, ensemble_r2]
})

print(eval_df.to_markdown(index=False))

# Visualisasi Komparasi Seluruh Model
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
metrics = ['RMSE', 'MAE', 'R-Squared (R2)']
bar_palettes = ['#ef4444', '#f59e0b', '#10b981']

for i, m in enumerate(metrics):
    sns.barplot(data=eval_df, x='Model', y=m, palette='Blues_r', ax=ax[i])
    ax[i].set_title(f'Perbandingan {m}', fontsize=12, fontweight='bold')
    ax[i].set_ylabel(m)
    ax[i].tick_params(axis='x', rotation=30)
    for p in ax[i].patches:
        ax[i].annotate(f"{p.get_height():.4f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig5_model_comparison.png'), dpi=300)
plt.show()

In [ ]:
# 2. Diagnostik Residual Error Model Ensemble
residuals = y_va - ensemble_val_preds

fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))

# Scatter Actual vs Predicted
sample_sub_idx = np.random.choice(len(y_va), size=5000, replace=False)
ax[0].scatter(y_va.iloc[sample_sub_idx], ensemble_val_preds[sample_sub_idx], alpha=0.25, color='#1d4ed8', s=10)
ax[0].plot([0, 1], [0, 1], color='#dc2626', linestyle='--', linewidth=2, label='Garis Sempurna (y = y_hat)')
ax[0].set_title('Scatter Actual vs. Predicted (Tri-Model Ensemble)', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Actual Utilization')
ax[0].set_ylabel('Predicted Utilization')
ax[0].legend()

# Histogram Residuals
sns.histplot(residuals, bins=50, kde=True, color='#059669', ax=ax[1])
ax[1].axvline(0, color='#dc2626', linestyle='--', linewidth=2)
ax[1].set_title('Distribusi Residual Error (y - y_hat)', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Residual Error')
ax[1].set_ylabel('Densitas')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig6_residual_diagnostics.png'), dpi=300)
plt.show()

In [ ]:
# 3. Analisis Kepentingan Fitur (Feature Importance LightGBM & CatBoost)
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model_lgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 7))
top_feats = feat_imp.head(15)
sns.barplot(data=top_feats, x='Importance', y='Feature', palette='mako')
plt.title('Top 15 Feature Importance (Model Boosting Split/Gain)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Fitur')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig7_feature_importance.png'), dpi=300)
plt.show()

print("Sepuluh Variabel Paling Berpengaruh:")
print(feat_imp.head(10).to_string(index=False))

# 6. Deployment & Strategic Recommendations (CRISP-DM Fase 6)

### 6.1 Rekomendasi Bisnis untuk Charge Point Operator (CPO) dan Pembuat Kebijakan
1. **Skema Tarif Dinamis (Dynamic Pricing):**
   * Model mengonfirmasi peningkatan utilisasi drastis pada jam sibuk (07.00 - 09.00 dan 16.00 - 19.00).
   * Penerapan diskon tarif pada jam santai (*off-peak hours*) dan tarif premium pada jam puncak mampu meratakan kurva beban jaringan (*load shifting*), meminimalkan waktu antrean, dan memaksimalkan pendapatan operator.
2. **Prioritas Penambahan Port Pengisian Cepat (DC Fast):**
   * Variabel daya (`power_output_kw`) dan ketersediaan charger ultra-cepat (> 150 kW) memiliki korelasi kuat terhadap utilisasi tinggi.
   * CPO disarankan memperluas kapasitas port di koridor jalan tol (*Highway Corridors*) dan pusat perbelanjaan (*Shopping Center*) yang konsisten mengalami okupansi di atas 75%.
3. **Penyediaan Ekosistem Fasilitas Terpadu:**
   * Keberadaan fasilitas seperti restoran (*dining*), WiFi, dan minimarket terbukti menaikkan retensi dan kenyamanan pengemudi selama proses pengisian daya.

### 6.2 Dampak Nyata terhadap Mitigasi Range Anxiety dan SDG 7
* **Mitigasi Range Anxiety:** Hasil prediksi utilisasi 30 menit ke depan dapat diintegrasikan melalui API ke sistem navigasi kendaraan listrik untuk mengarahkan pengguna ke stasiun terdekat yang tidak padat secara *real-time*.
* **Pencapaian SDG 7 (Affordable and Clean Energy):**
  * **Target 7.1:** Menjamin pemerataan akses energi hijau yang andal bagi pengguna kendaraan listrik publik dan pribadi.
  * **Target 7.2 & 7.3:** Mengarahkan konsumsi pengisian daya ke waktu ketersediaan energi surya/angin melimpah pada siang hari, mendukung efisiensi energi global.

### 6.3 Pembentukan File Submission Akhir
Menghasilkan berkas prediksi resmi berbasis **Tri-Model Ensemble** sesuai dengan spesifikasi lembar kompetisi ISFEST 2026:
* Format Berkas: `NamaTim_Submission.csv`
* Struktur Kolom: `id,utilization_rate`
* Integritas Baris: 263.550 baris dengan urutan ID yang persis sama dengan `sample_submission.csv`.

In [ ]:
# 1. Inferensi Prediksi Ketiga Model pada Data Uji
print("[INFO] Menjalankan inferensi data uji pada model LightGBM...")
test_preds_lgb = np.clip(model_lgb.predict(test_features[feature_cols]), 0.0, 1.0)

print("[INFO] Menjalankan inferensi data uji pada model CatBoost...")
test_preds_cb = np.clip(model_cb.predict(test_features[feature_cols]), 0.0, 1.0)

print("[INFO] Menjalankan inferensi data uji pada model XGBoost...")
test_preds_xgb = np.clip(model_xgb.predict(test_features[feature_cols]), 0.0, 1.0)

# 2. Penggabungan Ensemble Berbobot Optimal
test_preds_ensemble = np.clip(
    w_lgb * test_preds_lgb + w_cb * test_preds_cb + w_xgb * test_preds_xgb,
    0.0, 1.0
)

# 3. Penyusunan DataFrame Submission Sesuai Urutan Asli test_raw
sub_df = pd.DataFrame({
    'id': test_raw['id'],
    'utilization_rate': np.round(test_preds_ensemble, 3)
})

# 4. Verifikasi Integritas File terhadap sample_submission.csv
sample_sub_path = find_file('sample_submission.csv')
sample_sub = pd.read_csv(sample_sub_path)

assert len(sub_df) == len(sample_sub), f"Jumlah baris berbeda: {len(sub_df)} vs {len(sample_sub)}"
assert (sub_df['id'].values == sample_sub['id'].values).all(), "Urutan ID tidak identik dengan sample_submission.csv!"
assert sub_df['utilization_rate'].isnull().sum() == 0, "Ditemukan nilai kosong pada kolom prediksi!"

submission_file = f"{TEAM_NAME}_Submission.csv"
sub_df.to_csv(submission_file, index=False)

print("=== [BERHASIL] FILE SUBMISSION RESMI TERBENTUK ===")
print(f"Nama Berkas : {submission_file}")
print(f"Total Baris : {len(sub_df):,}")
print(f"Sebaran Nilai Prediksi:")
print(sub_df['utilization_rate'].describe())
print("\nLima Baris Pertama:")
print(sub_df.head())